# Week #5(a): Finding and Cropping an Object

[Open this notebook in Colab](https://colab.research.google.com/github/neurosynthetic/grant-vogt/blob/main/notebooks/week05a.ipynb)

> **Important:** Before you do any work, save your own copy to Google Drive.
>
> In Colab: **File → Save a copy in Drive**

In Week 4, you read information from a grid, transformed that information, and generated a new grid. This week, you will learn how to locate one object hiding inside a larger grid and remove the empty space around it.

The central idea is:

> **grid → object boundaries → cropped object**

You will also learn the two-input form of `range(...)`, which is useful when you want to visit only part of a list.

## Goals

By the end of this notebook, you will be able to:

- infer the rule of a new ARC-style task
- determine whether a row or column contains part of an object
- locate an object's top, bottom, left, and right boundaries
- extract a rectangular region from a grid
- combine small helper functions into `crop_object(grid)`
- explain the assumptions that make your transformation work

## Setup

Run the next cell once. It provides grid drawings and the **Check Exercise** buttons. If the runtime restarts, run this cell again.

In [ ]:
#@title Setup utilities (run once)

import copy
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from matplotlib.colors import ListedColormap

GRID_COLORS = [
    "#000000", "#0074D9", "#FF4136", "#2ECC40", "#FFDC00",
    "#AAAAAA", "#F012BE", "#FF851B", "#7FDBFF", "#870C25"
]
GRID_CMAP = ListedColormap(GRID_COLORS)


def draw_grid(ax, grid, title=""):
    rows = len(grid)
    cols = len(grid[0]) if rows else 0
    ax.imshow(grid, cmap=GRID_CMAP, vmin=0, vmax=9)

    font_size = max(6, min(12, 120 // max(rows, cols, 1)))
    colors_with_white_text = {0, 1, 2, 6, 9}
    for row_index, row in enumerate(grid):
        for column_index, color in enumerate(row):
            text_color = "white" if color in colors_with_white_text else "black"
            ax.text(
                column_index, row_index, str(color),
                ha="center", va="center",
                color=text_color, fontsize=font_size, fontweight="bold",
            )

    ax.set_title(title)
    ax.set_xticks([x - 0.5 for x in range(1, cols)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, rows)], minor=True)
    ax.grid(which="minor", color="white", linewidth=1)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)


def show_grid(grid, title="Grid"):
    _, ax = plt.subplots(figsize=(4, 3))
    draw_grid(ax, grid, title)
    plt.show()


def show_pair(input_grid, output_grid, label=""):
    _, axes = plt.subplots(1, 2, figsize=(8, 3))
    draw_grid(axes[0], input_grid, "Input")
    draw_grid(axes[1], output_grid, "Output")
    if label:
        plt.suptitle(label)
    plt.tight_layout()
    plt.show()


def show_examples(examples):
    for number, example in enumerate(examples, start=1):
        show_pair(example["input"], example["output"], f"Training example {number}")


def make_check_button(check_fn):
    button = widgets.Button(description="Check Exercise", button_style="info")
    output = widgets.Output()

    def on_click(_):
        output.clear_output()
        with output:
            check_fn()

    button.on_click(on_click)
    display(button, output)


print("Setup complete.")

# The task

We created this ARC-style task for the internship. It is not from the official ARC dataset.

Each input contains one colored object surrounded by black space. Study all three training examples before reading further.

In [ ]:
training_examples = [
    {
        "input": [
            [0,0,0,0,0,0,0,0],
            [0,0,2,2,0,0,0,0],
            [0,0,2,0,0,0,0,0],
            [0,0,2,2,2,0,0,0],
            [0,0,0,0,2,2,0,0],
            [0,0,0,0,0,0,0,0],
        ],
        "output": [
            [2,2,0,0],
            [2,0,0,0],
            [2,2,2,0],
            [0,0,2,2],
        ],
    },
    {
        "input": [
            [0,0,0,0,0,0],
            [0,0,0,0,0,0],
            [0,0,1,0,0,0],
            [0,1,1,1,0,0],
            [0,0,1,1,1,0],
            [0,0,0,1,0,0],
            [0,0,0,0,0,0],
        ],
        "output": [
            [0,1,0,0],
            [1,1,1,0],
            [0,1,1,1],
            [0,0,1,0],
        ],
    },
    {
        "input": [
            [0,0,0,0,0,0,3,0,0],
            [0,0,0,0,0,3,3,3,0],
            [0,0,0,0,3,3,0,3,3],
            [0,0,0,0,3,0,0,0,3],
            [0,0,0,0,0,0,0,0,0],
        ],
        "output": [
            [0,0,3,0,0],
            [0,3,3,3,0],
            [3,3,0,3,3],
            [3,0,0,0,3],
        ],
    },
]

show_examples(training_examples)

## Observation notebook

Answer these before coding:

1. What cells disappear from the input?
2. Does any part of the colored object disappear?
3. Does the object's pattern change?
4. Does its color change?
5. How would you describe the output's top edge?
6. How would you describe its left, right, and bottom edges?

### My rule

- **The black space:**
- **The colored object:**
- **The output boundaries:**
- **My rule in one sentence:**

## The task's format

Here are the assumptions our invented task follows:

- `0` is the black background.
- Every input contains exactly one colored object.
- Every part of the object uses the same nonzero color.
- Parts of the object touch horizontally or vertically, so it forms one connected shape.
- The output is the smallest rectangle that contains the complete object.
- Black cells inside that rectangle remain black.
- The object may touch an edge of the input grid.

The smallest rectangle containing an object is often called its **bounding box**.

# Part 1: Find occupied lines

Before finding the complete bounding box, learn to ask two smaller questions:

- Does this row contain any part of the object?
- Does this column contain any part of the object?

## Exercise 1: Check one row

Write `row_contains_object(row)` so it returns `True` when at least one cell is not `0`. Return `False` when the entire row is black.

As soon as you find a nonzero cell, you already know the answer and can return immediately.

In [ ]:
def row_contains_object(row):
    # Inspect each cell in the row.
    return False

In [ ]:
#@title Check Exercise 1

def check_exercise_1():
    tests = [
        ([0,0,0,0], False),
        ([0,0,2,0], True),
        ([7], True),
        ([0], False),
    ]
    try:
        for row, expected in tests:
            got = row_contains_object(row)
            if got != expected:
                print("Not yet. Return True if any cell is not 0.")
                print("Row:", row, "Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 1 complete: you can recognize an occupied row!")

make_check_button(check_exercise_1)

## Exercise 2: Check one column

Write `column_contains_object(grid, column_index)` so it returns `True` when the selected column contains at least one nonzero cell.

Scan the rows and inspect `row[column_index]` in each one.

In [ ]:
def column_contains_object(grid, column_index):
    # Inspect the selected cell in each row.
    return False

In [ ]:
#@title Check Exercise 2

def check_exercise_2():
    grid = training_examples[0]["input"]
    tests = [(0, False), (1, False), (2, True), (4, True), (5, True), (7, False)]
    try:
        for column_index, expected in tests:
            got = column_contains_object(grid, column_index)
            if got != expected:
                print("Not yet. Check the same column in every row.")
                print("Column:", column_index, "Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 2 complete: you can recognize an occupied column!")

make_check_button(check_exercise_2)

# Part 2: Find the object's boundaries

The bounding box has four boundaries:

- the first occupied row at the top
- the last occupied row at the bottom
- the first occupied column on the left
- the last occupied column on the right

We will find the row boundaries first and then use the same idea for columns.

## Exercise 3: Find the top and bottom rows

Write `find_row_bounds(grid)` so it returns a list containing `[top_row, bottom_row]`.

Start both variables at `-1`. Here, `-1` means “not found yet.”

Scan every row from top to bottom:

- The first occupied row becomes `top_row`.
- Every occupied row becomes the new `bottom_row`.
- By the end, `bottom_row` contains the last occupied row.

You may assume the grid contains an object.

In [ ]:
def find_row_bounds(grid):
    top_row = -1
    bottom_row = -1

    # Visit every row index from top to bottom.
    return [top_row, bottom_row]

In [ ]:
#@title Check Exercise 3

def check_exercise_3():
    tests = [
        (training_examples[0]["input"], [1, 4]),
        (training_examples[1]["input"], [2, 5]),
        (training_examples[2]["input"], [0, 3]),
        ([[0,0], [0,5], [0,0]], [1,1]),
    ]
    try:
        for grid, expected in tests:
            got = find_row_bounds(grid)
            if got != expected:
                print("Not yet. Find the first and last rows containing a nonzero cell.")
                print("Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 3 complete: you found the top and bottom boundaries!")

make_check_button(check_exercise_3)

## Exercise 4: Find the left and right columns

Write `find_column_bounds(grid)` so it returns `[left_column, right_column]`.

This follows the same pattern as `find_row_bounds`, but:

- visit column indices from left to right
- call `column_contains_object(grid, column_index)`
- use `len(grid[0])` to find the grid's width

In [ ]:
def find_column_bounds(grid):
    left_column = -1
    right_column = -1

    # Visit every column index from left to right.
    return [left_column, right_column]

In [ ]:
#@title Check Exercise 4

def check_exercise_4():
    tests = [
        (training_examples[0]["input"], [2,5]),
        (training_examples[1]["input"], [1,4]),
        (training_examples[2]["input"], [4,8]),
        ([[0,6,0]], [1,1]),
    ]
    try:
        for grid, expected in tests:
            got = find_column_bounds(grid)
            if got != expected:
                print("Not yet. Find the first and last columns containing a nonzero cell.")
                print("Got:", got, "Expected:", expected)
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 4 complete: you found the left and right boundaries!")

make_check_button(check_exercise_4)

## Reasoning checkpoint: Predict the output size

Suppose the boundaries are:

```python
top = 2
bottom = 5
left = 1
right = 4
```

Remember that both boundary rows and columns are included.

- What is the output height?
- What is the output width?
- Why do both calculations need `+ 1`?

### My prediction

- **Output height:**
- **Output width:**
- **Why add one:**

# Part 3: Extract the bounding box

Once the boundaries are known, copy only the cells inside them into a new grid.

## A new form of `range`

In Week 4, you used `range(n)` to produce numbers from `0` through `n - 1`.

The two-input form lets you choose where to start:

```python
range(start, stop)
```

It begins at `start` and stops **before** `stop`. For example, `range(2, 6)` produces `2, 3, 4, 5`.

Because our bottom and right boundaries must be included, use `bottom + 1` and `right + 1` as the stopping points.

## Exercise 5: Crop to known boundaries

Write `crop_grid(grid, top, bottom, left, right)` so it returns a new grid containing exactly those rows and columns.

Use one loop for row indices and another for column indices. Build one new row at a time. Do not change the input grid.

In [ ]:
def crop_grid(grid, top, bottom, left, right):
    cropped_grid = []
    # Copy the cells inside the four boundaries.
    return cropped_grid

In [ ]:
#@title Check Exercise 5

def check_exercise_5():
    tests = [
        (
            [[1,2,3], [4,5,6], [7,8,9]],
            1, 2, 0, 1,
            [[4,5], [7,8]],
        ),
        (
            [[1,2,3,4]],
            0, 0, 1, 2,
            [[2,3]],
        ),
        (
            [[1], [2], [3]],
            0, 2, 0, 0,
            [[1], [2], [3]],
        ),
    ]
    try:
        for grid, top, bottom, left, right, expected in tests:
            original = copy.deepcopy(grid)
            got = crop_grid(grid, top, bottom, left, right)
            if got != expected:
                print("Not yet. Copy every cell inside the inclusive boundaries.")
                print("Got:", got, "Expected:", expected)
                return
            if grid != original:
                print("The crop is correct, but the input grid changed.")
                return
    except Exception as exc:
        print("Your function raised an error:", exc)
        return
    print("Exercise 5 complete: you extracted a rectangular region!")

make_check_button(check_exercise_5)

# Part 4: Put everything together

Your helpers now form a complete transformation:

```text
                  ┌→ find_row_bounds ────┐
input grid ───────┤                      ├→ crop_grid → output grid
                  └→ find_column_bounds ─┘
```

## Final exercise: Write `crop_object`

Write `crop_object(grid)` using the three helpers:

1. Find the top and bottom rows.
2. Find the left and right columns.
3. Pass all four boundaries to `crop_grid`.

In [ ]:
def crop_object(grid):
    # Find both pairs of boundaries, then crop the grid.
    return None

In [ ]:
#@title Check the complete transformation

def check_crop_object():
    try:
        for number, example in enumerate(training_examples, start=1):
            original = copy.deepcopy(example["input"])
            got = crop_object(example["input"])
            if got != example["output"]:
                print(f"Training example {number} does not match yet.")
                return
            if example["input"] != original:
                print("The answer is right, but crop_object changed its input.")
                return

        hidden_input = [
            [0,0,0,0,0,0,0],
            [0,0,0,0,0,0,0],
            [0,0,0,6,6,0,0],
            [0,0,6,6,0,0,0],
            [0,0,6,0,0,0,0],
            [0,0,6,6,6,6,0],
        ]
        hidden_expected = [
            [0,6,6,0],
            [6,6,0,0],
            [6,0,0,0],
            [6,6,6,6],
        ]
        got = crop_object(hidden_input)
        if got != hidden_expected:
            print("The training examples pass, but a new test grid does not.")
            return
    except Exception as exc:
        print("Your transformation raised an error:", exc)
        return
    print("All checks pass. You built a complete cropping transformation!")

make_check_button(check_crop_object)

## Apply the transformation to a test input

This grid was not one of the training pairs. Before running `crop_object`, predict its four boundaries and output size.

In [ ]:
test_input = [
    [0,0,0,0,0,0,0,0],
    [0,0,0,0,0,0,0,0],
    [0,0,0,0,4,0,0,0],
    [0,0,0,4,4,4,0,0],
    [0,0,4,4,0,4,0,0],
    [0,0,0,4,4,4,0,0],
    [0,0,0,0,0,0,0,0],
]

show_grid(test_input, "Test input")

### My prediction

- **Top row:**
- **Bottom row:**
- **Left column:**
- **Right column:**
- **Output height:**
- **Output width:**

In [ ]:
test_output = crop_object(test_input)
show_pair(test_input, test_output, "Your transformation's answer")

# Stretch challenges

Choose any that interest you:

1. **Add padding:** Leave a one-cell black border around the cropped object.
2. **Crop and recolor:** Crop the object and change it to a requested color.
3. **Return the bounding box:** Write a function that returns all four boundaries in one list.
4. **Handle an empty grid:** Decide what `crop_object` should return when no object exists.
5. **Make a puzzle generator:** Place a small shape at a random location inside a larger black grid, then generate its cropped output.

## Wrap-up and reflection

This week, you located an object before transforming the grid. Answer briefly:

- Why were row and column helper functions useful?
- Which boundary was hardest to find?
- Why does `range(start, stop)` stop before `stop`?
- What assumption would break your current transformation?
- How might a bounding box help with a future ARC task?

### My reflection

- **The helpers helped because:**
- **Hardest boundary:**
- **Something I learned about `range`:**
- **One important assumption:**
- **A future use for bounding boxes:**

## Share your completed notebook

When you finish:

1. Confirm that all five exercise checks and the complete transformation check pass.
2. Complete the reflection.
3. In Colab, click **Share**.
4. Add **michael.d.moffitt@gmail.com** as a Viewer or Commenter.
5. Send the share invite.

Building the complete core transformation is the goal. The stretch challenges are optional.